# txt2mp3 on Colab (L4 GPU)

运行前: 菜单「运行时」→「更改运行时类型」→ 选 **L4 GPU**。按顺序运行每个单元格。
权重(~10 GB)每次新会话都会重新下载; MP3 保存到 Google Drive, 断线后不会丢。

Before running: menu "Runtime" → "Change runtime type" → select **L4 GPU**. Run each cell in order.
The weights (~10 GB) are re-downloaded in every new session; MP3 files are saved to Google Drive, so they survive a disconnect.

In [ ]:
# 1. 检查 GPU 与 CUDA 版本 (CUDA Version 需 >= 12.8)
# 1. Check GPU and CUDA version (CUDA Version must be >= 12.8)
!nvidia-smi
!ffmpeg -version | head -n 1

In [ ]:
# 2. 挂载 Google Drive (用于保存 MP3)
# 2. Mount Google Drive (used to store the MP3 files)
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/txt2mp3/mp3'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# 3. 拉取代码并安装依赖 (uv 会自动安装 Python 3.11)
# 3. Clone the code and install dependencies (uv installs Python 3.11 automatically)
!git clone https://github.com/luoliworks/txt2mp3.git /content/txt2mp3
%cd /content/txt2mp3
!pip install -q uv
!uv sync

In [ ]:
# 4. 下载 IndexTTS-2.5 权重 (~10 GB, 每次新会话都要下载)
# 4. Download the IndexTTS-2.5 weights (~10 GB, needed in every new session)
!uv run hf download IndexTeam/IndexTTS-2.5 --local-dir=checkpoints

In [ ]:
# 5. 提供 novel.txt (二选一)
# 5. Provide novel.txt (choose one)
# 方式 A: 本地上传
# Option A: upload from your computer
from google.colab import files
uploaded = files.upload()
import shutil
for name in uploaded:
    shutil.move(name, '/content/txt2mp3/novel.txt')
    break

# 方式 B: 从 Drive 复制 (取消注释并改路径, 同时注释掉上面方式 A)
# Option B: copy from Drive (uncomment and edit the path, and comment out Option A above)
# !cp /content/drive/MyDrive/txt2mp3/novel.txt /content/txt2mp3/novel.txt

In [ ]:
# 6. 检查 novel.txt
# 6. Check novel.txt
!pwd
!ls
!head -n 3 novel.txt

In [ ]:
# 7. 转 UTF-8 并拆分章节
# 7. Convert to UTF-8 and split into chapters
%cd /content/txt2mp3
!uv run python txt2utf8.py
!uv run python split_novel.py
!ls novel | head -n 20

In [ ]:
# 8. 转换为 MP3, 输出到 Drive (同名文件会自动备份为 -Copy)
# 8. Convert to MP3 and write to Drive (an existing file with the same name is backed up as -Copy)
%cd /content/txt2mp3
!uv run python txt2mp3.py --bf16 --output-dir {OUTPUT_DIR}



# 9. 转换结束后: 播放提示音 -> 等 5 分钟 -> 确保 Drive 写盘 -> 断开运行时 (停止计费)
# 9. After conversion: play a beep -> wait 5 minutes -> flush Drive -> disconnect the runtime (stops billing)
# 想取消自动断开: 在 5 分钟内点击本单元格的停止按钮
# To cancel the auto-disconnect, press this cell's stop button within the 5 minutes
import time
import numpy as np
from IPython.display import Audio, display
from google.colab import drive, runtime

sr = 22050
t = np.linspace(0, 0.3, int(sr * 0.3), endpoint=False)
beep = 0.4 * np.sin(2 * np.pi * 880 * t)
gap = np.zeros(int(sr * 0.2))
display(Audio(np.concatenate([beep, gap] * 5), rate=sr, autoplay=True))
print('转换完成, 5 分钟后自动断开运行时 / Done. The runtime disconnects in 5 minutes.')

time.sleep(300)

# 断开前才卸载 Drive, 这样 5 分钟内取消后仍可继续使用 Drive
# Unmount Drive only right before disconnecting, so Drive stays usable if you cancel within the 5 minutes
drive.flush_and_unmount()  # 确保 MP3 已写入 Drive / make sure MP3 files are written to Drive
runtime.unassign()